# Network Intrusion Detection — Progressive Dataset Evaluation

**Paper:** Chua & Salam (2023), *Evaluation of ML Algorithms in Network-Based Intrusion Detection Using Progressive Dataset*, Symmetry 15, 1251

**Setup:** Run this header cell first every time you open a new Colab session.

In [ ]:
# ── Header cell: run this first in every new Colab session ──────────────────
import sys, os

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repo so src/ modules are importable
REPO_URL = 'https://github.com/Rosette28/data-science-cyber-final-project'  # ← update
REPO_DIR = '/content/ids-project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 3. Install dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

print('Environment ready.')

In [ ]:
# ── Global configuration — only line you change between runs ─────────────────
DATA_DIR = '/content/drive/MyDrive/ids_data/raw/'  # ← set to your Drive folder

SEED = 42
SUBSAMPLE_FRAC = 0.10  # 10% of each day's CSV, read at load time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='tab10')

print(f'DATA_DIR = {DATA_DIR}')

---
## §1 — Data Loading & Initial Inspection

Load CIC-IDS2017 (train) and CSE-CIC-IDS2018 (progressive test), keeping ~10% via chunked reading. Inspect shape, dtypes, memory usage, column names, and temporal structure.

In [ ]:
from src.data_loading import load_cic2017, load_cic2018, align_schemas

df_train = load_cic2017(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_test  = load_cic2018(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_train, df_test = align_schemas(df_train, df_test)

print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)

### Shape, dtypes, memory, and column name inspection

In [ ]:
# ── Shape and memory ──────────────────────────────────────────────────────────
for name, df in [('Train (CIC-IDS2017)', df_train), ('Test  (CSE-CIC-IDS2018)', df_test)]:
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"{name}: {df.shape[0]:>7,} rows × {df.shape[1]} cols  |  {mem_mb:.1f} MB")

print()

# ── Dtype breakdown ───────────────────────────────────────────────────────────
print("Train dtype counts:")
print(df_train.dtypes.value_counts().to_string())
print("\nTest dtype counts:")
print(df_test.dtypes.value_counts().to_string())

In [ ]:
# ── Column name analysis ──────────────────────────────────────────────────────
# Features fall into five semantic groups derived from CICFlowMeter's documentation.
feature_groups = {
    'Packet length stats':    [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
    'Packet counts / rates':  [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
    'Inter-arrival times':    [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
    'TCP flags':              [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
    'Other / port / misc':    [c for c in df_train.columns if c not in sum([
        [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
        [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
        [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
        [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
        ['Label']
    ], []) and c != 'Label'],
}

print("Feature groups (76 features total after schema alignment):\n")
for group, cols in feature_groups.items():
    print(f"  {group} ({len(cols)}): {cols}")

print(f"\nLabel column: 'Label' — unique values in train: {df_train['Label'].unique()}")
print(f"                       — unique values in test:  {df_test['Label'].unique()}")

**Shape and memory** (after load + schema alignment, before deduplication):
- Train (CIC-IDS2017, 10% sample): ~281k rows × 77 cols, ~160 MB
- Test (CSE-CIC-IDS2018, 10% sample): ~125k rows × 77 cols, ~71 MB
- After hygiene cleaning (§1.3): **266,739 × 67 train / 106,906 × 67 test, 66 features**
- Both sets fit comfortably in free Colab RAM (~12 GB limit).

**Dtypes:** 52 `int64` (flag counts, raw packet/byte totals) · 24 `float32` (rates, means, stds — downcast from float64 to halve memory) · 1 `object` (Label).

**66 features across 5 semantic groups:**
- **Packet length stats:** min/max/mean/std of forward and backward packet sizes. Captures *what is being sent* — flooding attacks use fixed-size packets; exfiltration sends large payloads.
- **Packet counts / byte rates:** total packet counts, flow rates (bytes/s, packets/s), subflow counts. Captures *volume and asymmetry* — DoS shows extreme forward rate with near-zero backward.
- **Inter-arrival times (IAT):** min/max/mean/std of gaps between packets, plus Flow Duration. Captures *timing* — scans and floods have unusually small or regular IATs.
- **TCP flags:** SYN/FIN/RST/PSH/ACK/URG/ECE counts, initial TCP window sizes. Captures *connection behaviour* — SYN without ACK = SYN flood; window size fingerprints OS and botnet clients.
- **Other:** destination port, header lengths, active/idle time stats. Port alone is a strong discriminator — scanning generates traffic across unusual high ports.

**Labels:** 15 attack types + BENIGN in train; 10 attack types + BENIGN in test (2018 used integer encoding; 2017 had UTF-8 artifacts in Web Attack labels — both fixed at load time).

**Timestamp excluded from features:** Including raw calendar timestamps would cause time leakage ('2017 flow = benign, 2018 flow = attack'). IAT and Duration capture structural flow timing, not calendar position.


### Temporal structure and the progressive evaluation design

In [ ]:
import os, glob

# ── Per-day file breakdown for CIC-IDS2017 ────────────────────────────────────
cic2017_dir = os.path.join(DATA_DIR, 'cic2017')
cic2018_dir = os.path.join(DATA_DIR, 'cic2018')

print("CIC-IDS2017 source files (training set):")
for f in sorted(glob.glob(os.path.join(cic2017_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

print("\nCSE-CIC-IDS2018 source files (progressive test set):")
for f in sorted(glob.glob(os.path.join(cic2018_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

In [ ]:
# ── Attack type label distribution in each dataset ───────────────────────────
print("Attack types in TRAINING set (CIC-IDS2017):")
print(df_train['Label'].value_counts().to_string())

print("\nAttack types in TEST set (CSE-CIC-IDS2018):")
print(df_test['Label'].value_counts().to_string())

**Temporal structure — what time means in this project:**

**1. Within each dataset (daily granularity):**
CIC-IDS2017 is split across 8 CSV files covering five working days (Monday 3 July – Friday 7 July 2017). Monday is benign-only background traffic. From Tuesday onward, increasingly complex attacks are injected: Tuesday has brute-force (FTP-Patator, SSH-Patator); Wednesday has DoS/DDoS; Thursday has Web Attacks and Infiltration; Friday has DDoS and PortScan. CSE-CIC-IDS2018 arrives as a single pre-processed file — the within-day structure has been collapsed.

**2. Between datasets — the progressive gap (~8 months):**
All 2017 data precedes all 2018 data. There is zero temporal overlap. This is the core of the paper's methodology: a model trained on July 2017 traffic is evaluated on February–March 2018 traffic it has never seen.

**Class imbalance in both datasets:**
- Train: 226,117 BENIGN / 55,317 attacks → **80.4% benign**
- Test: 96,421 BENIGN / 28,475 attacks → **77.2% benign**

Both datasets reflect realistic network conditions where benign traffic dominates. This imbalance is a central methodological issue — the authors address it by downsampling to a 1:1 ratio before training.

**Attack type label distribution:**
Both datasets contain a mix of DoS, DDoS, brute-force, and bot traffic alongside the dominant BENIGN class. The two datasets use different naming conventions for some attack families (e.g. "DoS Hulk" in 2017 vs "DoS - Hulk" in 2018) — an artifact of how each dataset was labelled at collection time. For this project, all attacks are collapsed into a single binary ATTACK label for training and evaluation.

### Data hygiene scan

In [ ]:
# ── Duplicate rows ────────────────────────────────────────────────────────────
feat_cols = [c for c in df_train.columns if c != 'Label']

train_dups = df_train.duplicated(subset=feat_cols).sum()
test_dups  = df_test.duplicated(subset=feat_cols).sum()
print(f"Duplicate feature rows — train: {train_dups:,}  |  test: {test_dups:,}")

# ── Remaining NaN / inf (should be zero after _clean) ────────────────────────
train_nan = df_train[feat_cols].isnull().sum().sum()
test_nan  = df_test[feat_cols].isnull().sum().sum()
print(f"Remaining NaN values  — train: {train_nan}  |  test: {test_nan}")

train_inf = np.isinf(df_train[feat_cols].values).sum()
test_inf  = np.isinf(df_test[feat_cols].values).sum()
print(f"Remaining inf values  — train: {train_inf}  |  test: {test_inf}")

In [ ]:
# ── Constant / near-constant features (single unique value = useless) ─────────
constant_train = [c for c in feat_cols if df_train[c].nunique() <= 1]
constant_test  = [c for c in feat_cols if df_test[c].nunique()  <= 1]
print(f"Constant features in train: {constant_train or 'none'}")
print(f"Constant features in test:  {constant_test  or 'none'}")

# Near-constant: >99.9% of values are the same
near_const_train = [c for c in feat_cols
                    if df_train[c].value_counts(normalize=True).iloc[0] > 0.999]
print(f"\nNear-constant features in train (>99.9% one value): {near_const_train or 'none'}")

In [ ]:
# ── Drop duplicates and constant features; log decisions ─────────────────────
cols_to_drop = list(set(constant_train + constant_test))

df_train_clean = df_train.drop_duplicates(subset=feat_cols).reset_index(drop=True)
df_test_clean  = df_test.drop_duplicates(subset=feat_cols).reset_index(drop=True)

if cols_to_drop:
    df_train_clean = df_train_clean.drop(columns=cols_to_drop)
    df_test_clean  = df_test_clean.drop(columns=cols_to_drop)

print(f"After deduplication:")
print(f"  Train: {len(df_train):,} → {len(df_train_clean):,} rows  "
      f"(removed {len(df_train) - len(df_train_clean):,} duplicates)")
print(f"  Test:  {len(df_test):,}  → {len(df_test_clean):,}  rows  "
      f"(removed {len(df_test) - len(df_test_clean):,} duplicates)")
if cols_to_drop:
    print(f"\nDropped constant columns: {cols_to_drop}")
else:
    print("\nNo constant columns dropped.")

**Hygiene findings — cleaned shapes: 266,739 × 67 train, 106,906 × 67 test:**

- **Duplicates removed:** 14,695 from train (5.2%), 17,790 from test (14.3%). Common in network data — attack scripts hitting identical parameters, or benign apps opening repeated identical connections. Higher rate in test reflects the more uniform pre-processed 2018 file.
- **10 constant columns dropped:** `Bwd Avg Bulk Rate/Bytes/Packets`, `Fwd Avg Bulk Rate/Bytes/Bulk`, `Bwd PSH Flags`, `Bwd URG Flags`, `Fwd URG Flags`, `CWE Flag Count` — all zero across every row. Known CICFlowMeter limitation: bulk-transfer heuristics rarely trigger in lab-simulated traffic. Zero-variance features add only noise to a model.
- **Near-constant but kept:** `ECE Flag Count` and `RST Flag Count` exceed the 99.9% threshold in training but vary in the test set — rare non-zero values may still carry signal.
- Cleaned frames saved as `train_clean.joblib` / `test_clean.joblib` to Drive.


In [ ]:
# ── Save cleaned frames to Drive ──────────────────────────────────────────────
from joblib import dump

dump(df_train_clean, os.path.join(DATA_DIR, 'train_clean.joblib'))
dump(df_test_clean,  os.path.join(DATA_DIR, 'test_clean.joblib'))
print("Saved train_clean.joblib and test_clean.joblib to Drive.")

---
## §2 — Exploratory Data Analysis

Understand training and test distributions before feature engineering. Covers:
- **§2.1** Class distribution and the class-imbalance problem
- **§2.2** Feature distributions for key network-flow statistics
- **§2.3** Missing values verification
- **§2.4** Outlier analysis
- **§2.5** Temporal-feature analysis
- **§2.6** Cross-tabulation and group-by analysis
- **§2.7** Correlation analysis — method choice and justification (Spearman)
- **§2.8** Pre- vs post-balancing: visualising the 'symmetry' trade-off


In [ ]:
# §2 setup — reload cleaned frames if needed, define FIGURES_DIR
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import load as jload

try:
    df_train_clean
except NameError:
    df_train_clean = jload(os.path.join(DATA_DIR, 'train_clean.joblib'))
    df_test_clean  = jload(os.path.join(DATA_DIR, 'test_clean.joblib'))
    print('Reloaded cleaned frames from Drive.')

feat_cols_clean = [c for c in df_train_clean.columns if c != 'Label']
mask_benign = df_train_clean['Label'].str.strip().str.upper() == 'BENIGN'
print(f'Train: {df_train_clean.shape}, Test: {df_test_clean.shape}, Features: {len(feat_cols_clean)}')

FIGURES_DIR = str(pathlib.Path(DATA_DIR).parent / 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
sns.set_theme(style='whitegrid', palette='tab10')


### §2.1 — Class Distribution and Imbalance Analysis


In [ ]:
def binary_counts(df):
    b = df['Label'].apply(lambda x: 'BENIGN' if str(x).strip().upper()=='BENIGN' else 'ATTACK')
    return b.value_counts().reindex(['BENIGN','ATTACK'], fill_value=0)

train_bc = binary_counts(df_train_clean)
test_bc  = binary_counts(df_test_clean)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, bc) in zip(axes, [('Train (CIC-IDS2017)', train_bc),
                                   ('Test (CSE-CIC-IDS2018)', test_bc)]):
    bars = ax.bar(bc.index, bc.values, color=['steelblue','tomato'],
                  edgecolor='white', linewidth=1.2)
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('Row count')
    for bar, (lbl, val) in zip(bars, bc.items()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
                f'{val:,}\n({val/bc.sum()*100:.1f}%)',
                ha='center', va='bottom', fontsize=10)

plt.suptitle('Class distribution — before balancing (real prevalence)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_distribution_raw.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Train: attack-type breakdown ---')
print(df_train_clean['Label'].value_counts().to_string())
print('\n--- Test: attack-type breakdown ---')
print(df_test_clean['Label'].value_counts().to_string())


**Class distribution** *(Figure 1)*: 218,453 BENIGN vs 48,286 attack in train (**81.9% benign**); 89,937 vs 16,969 in test (**84.1% benign**). Both reflect typical enterprise network conditions.

**Attack-type drift — the core threat to the paper's conclusions:**

| Attack type | Train | Test | Change |
|-------------|-------|------|--------|
| Web Attack - XSS | 70 | Brute Force - XSS: **10,489** | ~150× amplification |
| PortScan | **14,444** | 0 | Disappears entirely |
| DDoS (generic) | 12,732 | HOIC: 3,415 / LOIC-UDP: 861 / LOIC-HTTP: 216 | Renamed & fragmented |
| Bot | 199 | 34 | Shrinks |
| Heartbleed | **1** | 0 | Effectively unlearnable |
| Infiltration | **6** | 0 | Effectively unlearnable |

A model trained on 70 XSS flows that then faces 10,489 XSS flows at test time is experiencing **concept drift**, not classical overfitting. The paper conflates the two — Phase 8.1 provides a direct test.

The authors' 1:1 downsampling also means accuracy on the balanced test set does not reflect deployment where 84% of inputs are benign. Quantified in §2.8.


### §2.2 — Feature Distributions


In [ ]:
KEY_FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Flow Bytes/s', 'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Fwd IAT Mean', 'Bwd IAT Mean', 'Flow IAT Mean',
]
KEY_FEATURES = [f for f in KEY_FEATURES if f in feat_cols_clean]

ncols = 3
nrows = (len(KEY_FEATURES) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()

for ax, feat in zip(axes, KEY_FEATURES):
    clip_val = df_train_clean[feat].quantile(0.99)
    for lbl, mask, color in [
        ('BENIGN', mask_benign, 'steelblue'),
        ('ATTACK', ~mask_benign, 'tomato')
    ]:
        ax.hist(df_train_clean.loc[mask, feat].clip(upper=clip_val),
                bins=50, alpha=0.5, color=color, label=lbl, density=True)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=9)
for ax in axes[len(KEY_FEATURES):]:
    ax.set_visible(False)

plt.suptitle('Feature distributions by class — train set (99th-percentile clipped)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()


**Feature distributions** *(Figure 2)*:

- All features are **heavily right-skewed** — most flows are short with few packets, with a heavy tail of high-volume flows. This violates Pearson's normality assumption and motivates Spearman (§2.7).
- **`Flow Duration`** (attack) is bimodal: spike near zero (fast DoS floods) + mass at ~10⁸ µs (slowloris/Hulk keeping connections open). Two completely different attack mechanics in one binary label.
- **`Bwd Packet Length Mean`** (attack) is also bimodal: spike at 0 (Slowhttptest — no server response) + mass at 1,500–2,000 bytes (DDoS response packets).
- **`Flow Bytes/s`** shows negative values — a CICFlowMeter edge-case artifact in this dataset.
- The per-feature overlap between benign and attack explains why no single feature suffices; the paper's 11-feature selection (§3) targets the most discriminative *combination*.


### §2.3 — Missing Values Verification


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, df) in zip(axes, [('Train', df_train_clean), ('Test', df_test_clean)]):
    nan_counts = df[feat_cols_clean].isnull().sum()
    nan_nz = nan_counts[nan_counts > 0]
    if nan_nz.empty:
        ax.text(0.5, 0.5, 'No missing values\n(cleaned in §1)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=13, color='seagreen')
    else:
        nan_nz.sort_values(ascending=False).plot(kind='bar', ax=ax, color='tomato')
        ax.set_ylabel('NaN count')
    ax.set_title(f'{name} — NaN per feature')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()

for name, df in [('Train', df_train_clean), ('Test', df_test_clean)]:
    numeric = df[feat_cols_clean].select_dtypes(include='number')
    inf_count = np.isinf(numeric).sum().sum()
    print(f'{name} — inf values remaining: {inf_count}')


### §2.4 — Outlier Analysis


In [ ]:
print('Outlier rate (IQR method, >Q3 + 1.5*IQR) — train set:\n')
outlier_summary = {}
for feat in KEY_FEATURES:
    q1, q3 = df_train_clean[feat].quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    n_out = int((df_train_clean[feat] > upper).sum())
    outlier_summary[feat] = n_out
    print(f'  {feat:<35}: {n_out:>6,}  ({n_out/len(df_train_clean)*100:.1f}%)')

top6 = sorted(outlier_summary, key=outlier_summary.get, reverse=True)[:6]
top6 = [f for f in top6 if f in df_train_clean.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), top6):
    cap = df_train_clean[feat].quantile(0.999)
    data = [
        df_train_clean.loc[mask_benign, feat].clip(upper=cap).values,
        df_train_clean.loc[~mask_benign, feat].clip(upper=cap).values,
    ]
    bp = ax.boxplot(data, tick_labels=['Benign', 'Attack'], patch_artist=True, showfliers=False)
    for patch, color in zip(bp['boxes'], ['steelblue', 'tomato']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle(
    'Box plots — highest-outlier features by class (99.9th-pct clipped, no fliers shown)',
    fontsize=11
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'outlier_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()


**Outlier findings** *(Figure 4)*: IQR outlier rates of 5–20% per feature — expected, not a quality problem.

- **DoS/DDoS:** outliers in `Flow Bytes/s` and IAT — flooding at maximum rate.
- **PortScan:** extreme values in packet-length features (tiny probes) and very short `Flow Duration`.
- **Benign elephant flows:** genuine outliers in byte-count features (large file transfers, video streaming).
- **Counterintuitive:** BENIGN has a *higher* median `Flow Bytes/s` than ATTACK (~210k vs ~100k bytes/s). Slow DoS attacks (slowloris, Slowhttptest) have near-zero byte rates, pulling the attack median down. `Flow Bytes/s` alone is not a reliable attack indicator.

**Decision — do not clip outliers:** tree models (DT, RF) handle them via split thresholds; SVM/ANN inputs will be scaled in §3. Clipping would destroy the attack-fingerprint patterns.


### §2.5 — Temporal Feature Analysis


In [ ]:
TEMPORAL_FEATS = [f for f in
    ['Flow Duration', 'Fwd IAT Mean', 'Bwd IAT Mean', 'Flow IAT Mean']
    if f in feat_cols_clean]

top10_labels = df_train_clean['Label'].value_counts().head(10).index.tolist()
df_sub = df_train_clean[df_train_clean['Label'].isin(top10_labels)]

fig, axes = plt.subplots(len(TEMPORAL_FEATS), 1, figsize=(13, 4 * len(TEMPORAL_FEATS)))
if len(TEMPORAL_FEATS) == 1:
    axes = [axes]

for ax, feat in zip(axes, TEMPORAL_FEATS):
    medians = df_sub.groupby('Label')[feat].median().sort_values()
    colors = ['steelblue' if 'BENIGN' in str(l).upper() else 'tomato'
              for l in medians.index]
    ax.barh(medians.index, medians.values, color=colors)
    ax.set_title(f'Median {feat} by class', fontsize=11)
    ax.set_xlabel(feat)
    ax.tick_params(labelsize=9)

plt.suptitle('Temporal features by attack class — top 10 classes, train set', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'temporal_features_by_class.png'), dpi=150, bbox_inches='tight')
plt.show()


**Temporal features by class** *(Figure 5)* — attack fingerprints align with known network behaviour:

- **DoS/DDoS** (GoldenEye, Hulk, LOIC/HOIC): near-zero IAT + short duration — flooding at maximum rate.
- **Slowloris / Slowhttptest:** longest flow durations (~10⁸ µs) — intentionally kept alive; near-zero Bwd IAT because server barely responds.
- **DoS Hulk:** high Fwd IAT despite being a DoS attack — Hulk sends valid HTTP GETs with pauses, not raw packet flooding.
- **PortScan:** near-zero duration and IAT (connection attempts fail immediately).
- **FTP-Patator / SSH-Patator:** moderate duration (full authentication protocol exchange per attempt).
- **BENIGN:** highest IAT variance — irregular, human-paced behaviour mixing DNS queries and streaming.

Note: DoS attacks dominate the axis scale — BENIGN, Bot, and PortScan bars are nearly invisible in Figure 5. This is a scale limitation, not a data problem.

**Timestamp excluded:** including calendar timestamps would teach the model '2017 = benign, 2018 = attack' (time leakage). IAT and Duration capture structural flow timing and are safe.


### §2.6 — Cross-tabulation and Group-by Analysis


In [ ]:
SUMMARY_FEATS = [f for f in
    ['Flow Duration','Total Fwd Packets','Total Backward Packets',
     'Flow Bytes/s','Fwd Packet Length Mean','Bwd Packet Length Mean']
    if f in feat_cols_clean]

top8 = df_train_clean['Label'].value_counts().head(8).index.tolist()
group_stats = (
    df_train_clean[df_train_clean['Label'].isin(top8)]
    .groupby('Label')[SUMMARY_FEATS]
    .median()
    .round(2)
)
print('Median feature values by class (top 8, train set):')
print(group_stats.to_string())
print()

if 'Total Fwd Packets' in feat_cols_clean and 'Total Backward Packets' in feat_cols_clean:
    ratio = (df_train_clean['Total Backward Packets'] /
             (df_train_clean['Total Fwd Packets'] + 1e-6)).clip(0, 100)
    ratio_by_class = (
        df_train_clean[df_train_clean['Label'].isin(top8)]
        .assign(_ratio=ratio)
        .groupby('Label')['_ratio']
        .median()
        .sort_values()
    )
    print('Median Bwd/Fwd packet ratio by class:')
    print('  (0 = purely unidirectional DoS;  ~1 = balanced bidirectional traffic)')
    print(ratio_by_class.to_string())


**Bwd/Fwd packet ratio by class** — confirms known network-security patterns:

| Class | Median Bwd/Fwd | Note |
|-------|---------------|------|
| DoS Slowhttptest | **0.000** | Purely unidirectional — server never responds |
| DoS slowloris | 0.214 | Mostly unidirectional — connection starved |
| DoS GoldenEye | 0.625 | HTTP requests get some responses before slowdown |
| DDoS | 0.750 | Victim responds to a fraction of the flood |
| DoS Hulk | 0.857 | Near-symmetric — server tries to answer valid HTTP GETs |
| BENIGN | **1.000** | Perfectly bidirectional (TCP/HTTP/DNS) |
| PortScan | **1.000** | CICFlowMeter captures SYN+SYN-ACK → appears symmetric |
| FTP-Patator | **1.667** | *Server-heavy:* multiple 530 failure messages per login attempt |

- **FTP-Patator > 1** is the most unusual finding: attacker-initiated traffic generates *more server-side* packets than client-side — the FTP server sends banners, challenges, and error codes for every failed attempt.
- **PortScan = 1** is counterintuitive: CICFlowMeter captures completed SYN+SYN-ACK flows, not raw SYN probes — so packet-direction features alone won't easily distinguish scans from benign traffic.


### §2.7 — Correlation Analysis — Method Choice and Justification


**Why Spearman (not Pearson or Kendall):**

| Method | Key assumption | Verdict for this dataset |
|--------|---------------|-------------------------|
| **Pearson** | Linear relationship; normally distributed, homoscedastic data | ❌ Network features follow power-law distributions; outliers are attack signals, not errors; relationship is monotonic but not strictly linear |
| **Kendall** | Rank-based; no normality assumption; robust to outliers | ✅ Correct assumptions, but **O(n²)** computation — impractical for ~267k rows and 66 features |
| **Spearman** | Rank-based; no normality assumption; robust to outliers | ✅ Captures monotonic relationships; **O(n log n)**; correct and efficient for this scale |

**Practical vs. statistical significance:** With ~267k rows, virtually every non-zero correlation is statistically significant (p < 0.001). What matters is *practical* significance: |r| > 0.90 identifies feature pairs carrying essentially redundant information — candidates for removal in §3 feature selection.


In [ ]:
# Spearman on a 10k-row subsample — fast and representative
_CORR_N = 10_000
df_corr_sample = df_train_clean[feat_cols_clean].sample(
    min(_CORR_N, len(df_train_clean)), random_state=SEED
)
print(f'Computing Spearman matrix on {len(df_corr_sample):,} rows x {len(feat_cols_clean)} features...')
corr_matrix = df_corr_sample.corr(method='spearman')
print('Done.')

# Pairs with |r| > 0.90 — redundant features
high_pairs = []
cols = corr_matrix.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = float(corr_matrix.iloc[i, j])
        if abs(r) > 0.90:
            high_pairs.append((abs(r), r, cols[i], cols[j]))
high_pairs.sort(reverse=True)

print(f'\nFeature pairs |Spearman r| > 0.90  ({len(high_pairs)} total — top 20 shown):')
for _, r, c1, c2 in high_pairs[:20]:
    print(f'  r={r:+.3f}  {c1}  <->  {c2}')


In [ ]:
# Spearman heatmap — top 30 highest-variance features, hierarchically clustered
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

top30 = df_corr_sample.var().nlargest(30).index.tolist()
sub_corr = corr_matrix.loc[top30, top30]

try:
    dist_mat = (1 - sub_corr.abs()).clip(lower=0)
    np.fill_diagonal(dist_mat.values, 0.0)
    link = linkage(squareform(dist_mat.values), method='average')
    order = leaves_list(link)
    sub_corr = sub_corr.iloc[order, order]
except Exception as e:
    print(f'Hierarchical clustering skipped ({e}); using original order.')

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(sub_corr, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, cbar_kws={'label': 'Spearman r'},
            xticklabels=True, yticklabels=True)
ax.tick_params(axis='x', labelrotation=45, labelsize=7)
ax.tick_params(axis='y', labelrotation=0, labelsize=7)
ax.set_title('Spearman correlation — top 30 highest-variance features (train set)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'spearman_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()


**Spearman correlation** *(Figure 6)*: 99 pairs with |r| > 0.90. Ten pairs are r = 1.000 (mathematically identical):

| Pair (r = 1.000) | Reason |
|-----------------|--------|
| `Subflow Fwd/Bwd Packets/Bytes` ↔ `Total Fwd/Bwd Packets/Length` (4 pairs) | CICFlowMeter: Subflow = Total for single-subflow flows |
| `Avg Fwd/Bwd Segment Size` ↔ `Fwd/Bwd Packet Length Mean` (2 pairs) | Two names for the same computation |
| `Packet Length Std` ↔ `Packet Length Variance` | Mathematical identity: Var = Std² |
| `Idle Max` ↔ `Idle Mean` | Near-constant idle times |
| `Fwd PSH Flags` ↔ `SYN Flag Count` | CIC capture artefact |
| `ECE Flag Count` ↔ `RST Flag Count` | CIC capture artefact |

**Three redundancy clusters (visible in Figure 6):**
1. **Subflow cluster:** four subflow features are exact duplicates of four total-flow features → all four should be dropped.
2. **Packet length / segment size:** `Avg Segment Size` = `Packet Length Mean`; `Packet Length Variance` = `Packet Length Std²` → keep one per pair.
3. **IAT cluster:** `Bwd/Fwd/Flow IAT Max/Mean/Total` all r ≥ 0.997 with each other → one representative suffices.

The flag artefacts (PSH↔SYN, ECE↔RST) are CIC-specific — if these correlations break in a different network capture, a model relying on them will degrade, contributing to the cross-dataset performance drop.

This directly motivates the authors' **66 → 11 feature reduction** reproduced in §3.


### §2.8 — Pre- vs Post-Balancing: the 'Symmetry' Trade-off


In [ ]:
df_bin = df_train_clean.assign(
    _binary=df_train_clean['Label'].apply(
        lambda x: 'BENIGN' if str(x).strip().upper()=='BENIGN' else 'ATTACK'
    )
)
before = df_bin['_binary'].value_counts().reindex(['BENIGN','ATTACK'], fill_value=0)

n_min = int(before.min())
balanced = pd.concat([
    df_bin[df_bin['_binary']==cls].sample(n=n_min, random_state=SEED)
    for cls in ['BENIGN','ATTACK']
])
after = balanced['_binary'].value_counts().reindex(['BENIGN','ATTACK'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (title, bc) in zip(axes, [
    ('Before balancing — real prevalence', before),
    ("After 1:1 downsampling (paper's 'symmetry')", after)
]):
    bars = ax.bar(bc.index, bc.values, color=['steelblue','tomato'],
                  edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Row count')
    total = bc.sum()
    for bar, val in zip(bars, bc.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                f'{val:,}\n({val/total*100:.1f}%)',
                ha='center', va='bottom', fontsize=10)

plt.suptitle("Class distribution: real prevalence vs. paper's 1:1 'symmetry'", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_balancing_effect.png'), dpi=150, bbox_inches='tight')
plt.show()

discarded = int(before.sum() - after.sum())
print(f'Rows discarded by 1:1 downsampling: {discarded:,}')
print(f'Training data retained: {after.sum()/before.sum()*100:.1f}%')


**The real cost of 1:1 'symmetry'** *(Figure 7)*: keeping only 48,286 of 218,453 BENIGN rows means **170,167 rows discarded — 63.8% of all training data**. Only 36.2% is used.

- **Metric inflation:** accuracy on a 1:1 balanced test set is systematically higher than on a deployment-realistic 84%-benign set. The paper's reported ~96–99% in-distribution accuracy is an upper bound, not a deployment estimate.
- **Data waste:** the model sees a less representative sample of legitimate traffic — rare-but-legitimate flows that fall in the discarded majority may become false positives.
- **Alternatives not explored:** class-weighted loss, SMOTE (synthetic oversampling), or threshold tuning — all address imbalance without discarding 63.8% of the data. The paper's 'symmetry' framing elevates a pragmatic compromise to a principle it doesn't warrant.

Phase 8.2 re-evaluates all models on the real-prevalence test split to measure the accuracy gap directly.


---
## §3 — Feature Engineering

Reproduce the authors' preprocessing pipeline, then add feature creation,
scaling, feature selection (RF importance + brute-force add-one loop),
and a redundancy analysis. Steps:

- **§3.1** Cleaning recap — mirrors the authors' `01_Dataset_Preprocessing.ipynb`
- **§3.2** Binary relabelling — collapse 15 attack types → `ATTACK`
- **§3.3** Class balancing — 1:1 downsample (paper's method) + preserve real-prevalence copy
- **§3.4** Categorical encoding — choice and justification
- **§3.5** Derived features — four engineered flow metrics with cyber rationale
- **§3.6** Feature scaling — StandardScaler fitted on balanced training set
- **§3.7** Feature selection — RF importance ranking + brute-force add-one accuracy curve
- **§3.8** Redundancy analysis — correlation clusters, RF importance, VIF


In [ ]:
# §3 setup — reload cleaned frames if needed; import preprocessing module
import os, pathlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load as jload

try:
    df_train_clean
except NameError:
    df_train_clean = jload(os.path.join(DATA_DIR, 'train_clean.joblib'))
    df_test_clean  = jload(os.path.join(DATA_DIR, 'test_clean.joblib'))
    print('Reloaded cleaned frames from Drive.')

from src.preprocessing import (
    binary_relabel, balance_1to1, add_derived_features, get_feature_cols,
    fit_scaler, apply_scaler, select_features_rf, brute_force_select, compute_vif,
)

FIGURES_DIR = str(pathlib.Path(DATA_DIR).parent / 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
sns.set_theme(style='whitegrid', palette='tab10')

print(f'Train: {df_train_clean.shape}  |  Test: {df_test_clean.shape}')
print(f'Features available: {len(get_feature_cols(df_train_clean))}')


### §3.1 — Cleaning Recap — mirroring the authors


In [ ]:
# §3.1 — Cleaning was performed in §1; document it here for the reproducibility record.
print('=== Cleaning steps applied in §1 (mirrors 01_Dataset_Preprocessing.ipynb) ===\n')
steps = [
    ('1. Strip column-name whitespace',
     'Identical to authors — CICFlowMeter produces inconsistent spacing'),
    ('2. Drop Fwd Header Length.1',
     'Duplicate of Fwd Header Length; present only in CIC-IDS2017'),
    ('3. Replace ±inf → NaN, dropna()',
     'Identical to authors — removes ~0.05% train rows, ~0.63% test rows'),
    ('4. Downcast float64 → float32',
     'Our addition — halves RAM on free Colab; negligible precision loss'),
    ('5. Drop duplicate feature rows',
     'Our addition — removes 5.2% train / 14.3% test; reduces noise from repeated flows'),
    ('6. Drop 10 constant columns',
     'Our addition — zero-variance features (CICFlowMeter bulk-rate artefacts)'),
]
for step, note in steps:
    print(f'  {step}\n    → {note}\n')
print(f'Resulting shapes:  train {df_train_clean.shape}  |  test {df_test_clean.shape}')
print(f'Features: {len(get_feature_cols(df_train_clean))}  (down from 78 in raw CSVs)')


**Cleaning decisions match the authors** for the core three steps (strip names, drop duplicate column,
remove inf/NaN). Two extra steps were added — `float32` downcasting and deduplication — to fit
free Colab RAM and reduce noise. Neither changes statistical structure: deduplication removes
exact-copy rows (no additional information), and float32 rounding error is below 1e-7.


### §3.2 — Binary Relabelling


In [ ]:
# §3.2 — Collapse 15+ attack types → binary ATTACK / BENIGN.
# Mirrors the authors' 01_Dataset_Preprocessing.ipynb Step 4.
df_train_bin = binary_relabel(df_train_clean)
df_test_bin  = binary_relabel(df_test_clean)

for name, df_orig, df_bin in [
    ('Train', df_train_clean, df_train_bin),
    ('Test',  df_test_clean,  df_test_bin),
]:
    orig_classes = df_orig['Label'].nunique()
    vc = df_bin['Label'].value_counts()
    print(f'{name}: {orig_classes} original classes → 2 binary classes')
    print(f'  BENIGN : {vc["BENIGN"]:>7,}  ({vc["BENIGN"]/len(df_bin)*100:.1f}%)')
    print(f'  ATTACK : {vc["ATTACK"]:>7,}  ({vc["ATTACK"]/len(df_bin)*100:.1f}%)')
    print()


**Binary relabelling rationale:** The paper collapses all attack types to a single `malicious` label
to focus on the binary detection task. This simplifies the target and allows a fair 6-model comparison
without multiclass complications.

**Trade-off:** binary collapse discards within-class structure. As shown in §2.1, XSS in train (n=70)
and XSS in test (n=10,489) are statistically very different — binary collapse hides this distribution shift.
The original multiclass `Label` is preserved in `df_train_clean` / `df_test_clean`
for the optional multiclass extension (Phase 8.3).


### §3.3 — Class Balancing (1:1) and Real-Prevalence Copy


In [ ]:
# §3.3 — Downsample BENIGN to 1:1; keep a real-prevalence copy for Phase 8.2.
# Only the training set is balanced; the test set stays at real prevalence.
df_train_bal, df_train_imbal = balance_1to1(df_train_bin, seed=SEED)

print('=== Training set: before and after 1:1 balancing ===')
for label, df in [('Real prevalence (Phase 8.2 copy)', df_train_imbal),
                   ('1:1 balanced    (mirrors paper)',  df_train_bal)]:
    vc = df['Label'].value_counts()
    total = len(df)
    print(f'\n  {label}')
    print(f'    BENIGN: {vc["BENIGN"]:>7,}  ({vc["BENIGN"]/total*100:.1f}%)')
    print(f'    ATTACK: {vc["ATTACK"]:>7,}  ({vc["ATTACK"]/total*100:.1f}%)')
    print(f'    Total:  {total:>7,}')

discarded = len(df_train_imbal) - len(df_train_bal)
print(f'\nRows discarded: {discarded:,} ({discarded/len(df_train_imbal)*100:.1f}% of training data)')
print('Test set: kept at real prevalence — not balanced.')


**Two copies kept deliberately:**

- `df_train_bal` — 1:1 balanced copy used for all model training and in-distribution
  evaluation (mirrors the paper). The 50/50 ratio means a trivial classifier achieves 50% accuracy,
  so any model above that is doing real work.
- `df_train_imbal` — real-prevalence copy (~82% BENIGN) used in Phase 8.2 to re-score
  models under deployment conditions and challenge the paper's "symmetry" framing.

**Critique note (→ report §2):** The paper discards ~63.8% of training data to achieve 1:1.
Alternatives — `class_weight='balanced'`, SMOTE oversampling, or decision-threshold tuning —
would address imbalance without discarding majority-class information.
The paper does not compare these alternatives.


### §3.4 — Categorical Encoding — Choice and Justification


In [ ]:
# §3.4 — Check for non-numeric feature columns; discuss Destination Port encoding.
feat_cols = get_feature_cols(df_train_bal)

dtype_counts = df_train_bal[feat_cols].dtypes.value_counts()
print('Feature dtype breakdown (balanced training set):')
print(dtype_counts.to_string())

non_numeric = df_train_bal[feat_cols].select_dtypes(exclude='number').columns.tolist()
print(f'\nNon-numeric feature columns: {non_numeric or "none — all features are numeric"}')

# Destination Port — the integer feature that warrants encoding discussion
port_col = 'Destination Port'
if port_col in df_train_bal.columns:
    port = df_train_bal[port_col]
    print(f'\nDestination Port: dtype={port.dtype}, unique={port.nunique():,}, '
          f'min={port.min():.0f}, max={port.max():.0f}')
    print(f'Top-10 most frequent ports in balanced train:')
    print(port.value_counts().head(10).to_dict())
    well_known = (port < 1024).sum()
    registered = ((port >= 1024) & (port < 49152)).sum()
    dynamic    = (port >= 49152).sum()
    print(f'\nPort range breakdown:  well-known 0-1023: {well_known:,}  '
          f'registered 1024-49151: {registered:,}  dynamic 49152+: {dynamic:,}')


**Encoding decision: treat `Destination Port` as a continuous numeric integer (no encoding applied).**

All 66 features are already numeric after §1 cleaning. The only encoding question is `Destination Port`:

| Option | Problem |
|--------|---------|
| **One-hot** | Up to 65,536 unique ports → impractical sparse columns |
| **Label/ordinal** | Arbitrary ordering (port 443 is not numerically "larger" than port 80 in a meaningful sense) |
| **Port-range bins** | Loses within-bin discrimination (port 22 vs port 80 collapse to one category) |
| **Keep as integer (chosen)** | Port ranges have genuine ordinal structure; tree models learn port-specific splits (`port ≤ 1023`); SVM/ANN treat it as a feature on equal scale with others after StandardScaling |

The paper treats `Destination Port` as numeric throughout — it appears in their final 11-feature set.
**Limitation for generalisation:** port-specific patterns from CIC-IDS2017 (e.g., scans targeting
port 80/443) may not transfer to CIC-IDS2018 if attack tools target different ports —
contributing to the cross-dataset performance drop.


### §3.5 — Derived Features


In [ ]:
# §3.5 — Add four engineered flow features.
# Applied to all three sets independently (no target leakage, no cross-set statistics).
df_train_bal   = add_derived_features(df_train_bal)
df_train_imbal = add_derived_features(df_train_imbal)
df_test_bin    = add_derived_features(df_test_bin)

derived = ['feat_bwd_fwd_ratio', 'feat_pkt_len_range', 'feat_bytes_per_pkt', 'feat_win_ratio']
print('New derived feature stats (balanced train set):\n')
print(df_train_bal[derived].describe().round(3).to_string())
print('\nMedian by class (balanced train):')
print(df_train_bal.groupby('Label')[derived].median().round(3).to_string())


**Four derived features and their network-security rationale:**

| Feature | Formula | Cyber meaning |
|---------|---------|---------------|
| `feat_bwd_fwd_ratio` | Bwd pkts / Fwd pkts | 0 = purely unidirectional (slowloris, Slowhttptest). ~1 = normal TCP. >1 = server-heavy (FTP-Patator). Compresses the §2.6 Bwd/Fwd analysis into one scalar. |
| `feat_pkt_len_range` | Max pkt − Min pkt | 0 = all packets identical (DDoS fixed-size UDP floods). Large = size-varied legitimate or exfiltration traffic. |
| `feat_bytes_per_pkt` | Flow Bytes/s ÷ Flow Pkts/s | Time-normalised payload density. Near-zero = slow-rate DoS (keep-alive packets). Large = bulk data transfer. |
| `feat_win_ratio` | Init_Win_fwd / Init_Win_bwd | TCP window asymmetry. Botnet clients and scanner tools advertise atypical window sizes vs. real OS stacks (64 k default). |

Feature selection in §3.7 empirically determines whether these features are useful — if RF ranks them
low, they won't enter the final feature set.


### §3.6 — Feature Scaling


In [ ]:
# §3.6 — Fit StandardScaler on the balanced training set;
# apply to the imbalanced training copy and the test set.
feat_cols = get_feature_cols(df_train_bal)

X_bal   = df_train_bal[feat_cols]
y_bal   = df_train_bal['Label']
X_imbal = df_train_imbal[feat_cols]
y_imbal = df_train_imbal['Label']
X_test  = df_test_bin[feat_cols]
y_test  = df_test_bin['Label']

scaler, X_bal_scaled = fit_scaler(X_bal)
X_imbal_scaled = apply_scaler(scaler, X_imbal)
X_test_scaled  = apply_scaler(scaler, X_test)

print('Scaler fitted on balanced training set only (no leakage from imbal/test).')
print(f'X_bal_scaled   shape: {X_bal_scaled.shape}  '
      f'mean_max_abs: {X_bal_scaled.mean().abs().max():.5f}  '
      f'std_mean: {X_bal_scaled.std().mean():.5f}')
print(f'X_imbal_scaled shape: {X_imbal_scaled.shape}')
print(f'X_test_scaled  shape: {X_test_scaled.shape}')
print()
print('Sample feature means/stds on scaled training set:')
for c in feat_cols[:4]:
    print(f'  {c:<45} mean={X_bal_scaled[c].mean():+.4f}  std={X_bal_scaled[c].std():.4f}')


**Scaling rationale:**

- **SVM (RBF kernel):** distance-based — `Destination Port` (0–65 535) would dominate
  `FIN Flag Count` (0–5) without scaling, making the SVM effectively ignore low-range features.
- **ANN/DNN (MLPClassifier):** gradient descent converges faster and more reliably
  with zero-mean, unit-variance inputs.
- **DT/RF/NB:** insensitive to scale — scaling changes nothing for these models,
  but applying it uniformly simplifies the pipeline.

**Implementation note:** the scaler is fitted **only on the balanced training set**
and then applied to the imbalanced copy and the test set.
Fitting on test data would leak the test distribution into the training normalisation — a data-leakage bug.


### §3.7 — Feature Selection — RF Importance + Brute-Force Add-One


In [ ]:
# §3.7a — Stage 1: RF importance ranking on the balanced, scaled training set.
# Mirrors the authors' 02_Feature_Selection.ipynb Stage 1.
print('Fitting RF for feature importance ranking...')
t0 = time.time()
top20_features, full_importance = select_features_rf(
    X_bal_scaled, y_bal, top_n=20, seed=SEED
)
print(f'Done in {time.time()-t0:.1f}s\n')

print('Top-20 features by RF importance:')
for rank, feat in enumerate(top20_features, 1):
    imp_val = full_importance[feat]
    print(f'  {rank:2d}. {feat:<45} {imp_val:.4f}')


In [ ]:
# §3.7b — Feature importance bar chart (top 20)
fig, ax = plt.subplots(figsize=(10, 7))
imp_top20 = full_importance.head(20).sort_values(ascending=True)

palette = sns.color_palette('Blues_d', len(imp_top20))
bars = ax.barh(imp_top20.index, imp_top20.values, color=palette, edgecolor='none')

ax.set_xlabel('Mean impurity decrease (RF Gini importance)', fontsize=11)
ax.set_title('Feature importance — top 20 (RF, 100 trees, balanced train set)', fontsize=12)
ax.tick_params(axis='y', labelsize=9)
for bar, val in zip(bars, imp_top20.values):
    ax.text(val + 0.0002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_importance_rf.png'), dpi=150, bbox_inches='tight')
plt.show()


**RF importance results** *(Figure 8)*: Features are dominated by **packet-length statistics**,
matching the paper's finding (their top-11 is also packet-length-heavy).

Why packet-length features dominate:
- **DoS flooding:** fixed-size packets → `Packet Length Std ≈ 0`, `Max Pkt ≈ Min Pkt`
- **Normal web traffic:** mixed small ACKs + large HTTP payloads → high variance
- **Port scanning:** tiny SYN probes → very small `Max Packet Length`

`Destination Port` and TCP window features (`Init_Win_bytes_*`) appear because scan targets
and botnet clients are port/window-specific in the 2017 dataset — a dataset-specific pattern
that may not generalise to 2018.


In [ ]:
# §3.7c — Stage 2: brute-force add-one-feature accuracy curve.
# Mirrors the authors' 02_Feature_Selection.ipynb Stage 2.
# Uses LinearSVC/NaiveBayes/MLP on a 10k subsample, 3-fold CV.
# Expected runtime: ~3–5 min on Colab CPU.
print('Running brute-force feature selection (~3–5 min)...')
t0 = time.time()
bf_results = brute_force_select(
    X_bal_scaled, y_bal,
    ranked_features=top20_features,
    max_features=20,
    subsample_n=10_000,
    cv=3,
    seed=SEED,
)
print(f'\nDone in {time.time()-t0:.1f}s\n')
print(bf_results.to_string(index=False, float_format='{:.4f}'.format))


In [ ]:
# §3.7d — Accuracy vs. #features curve
fig, ax = plt.subplots(figsize=(11, 5))
color_map  = {'LinearSVC': 'steelblue', 'NaiveBayes': 'tomato', 'MLP': 'seagreen'}
marker_map = {'LinearSVC': 'o', 'NaiveBayes': 's', 'MLP': '^'}

for model in ['LinearSVC', 'NaiveBayes', 'MLP']:
    ax.plot(bf_results['n_features'], bf_results[model],
            label=model, color=color_map[model], marker=marker_map[model],
            linewidth=2, markersize=6)

ax.set_xlabel('Number of features (added in RF-importance order)', fontsize=11)
ax.set_ylabel('3-fold CV accuracy (10 k-row subsample)', fontsize=11)
ax.set_title('Feature selection — brute-force add-one accuracy curve', fontsize=12)
ax.legend(fontsize=10)
ax.set_xticks(bf_results['n_features'])
ax.grid(True, alpha=0.4)
ax.set_ylim(0.5, 1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_selection_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print plateau point for each model
for model in ['LinearSVC', 'MLP']:
    accs = bf_results[model].values
    diffs = np.diff(accs)
    plateau = [n + 2 for n in range(len(diffs)) if all(abs(d) < 0.005 for d in diffs[n:])]
    if plateau:
        print(f'{model}: plateau at n={plateau[0]}  (acc={accs[plateau[0]-1]:.4f})')


In [ ]:
# §3.7e — Finalise the selected feature set.
# Inspect bf_results + the plot above, then set OPTIMAL_N to the plateau.
# Default: 11, matching the authors. Adjust after running if your curve differs.
OPTIMAL_N = 11

selected_features = top20_features[:OPTIMAL_N]

print(f'Selected {OPTIMAL_N} features (RF importance order):')
for i, feat in enumerate(selected_features, 1):
    print(f'  {i:2d}. {feat}')

# Rebuild scaled X matrices with only the selected features
X_bal_sel   = X_bal_scaled[selected_features]
X_imbal_sel = X_imbal_scaled[selected_features]
X_test_sel  = X_test_scaled[selected_features]

print(f'\nX_bal_sel:   {X_bal_sel.shape}')
print(f'X_imbal_sel: {X_imbal_sel.shape}')
print(f'X_test_sel:  {X_test_sel.shape}')


**Feature selection results** *(Figure 9)*:

The accuracy curve shows a clear elbow: accuracy rises steeply from n=1 to ~n=5–8,
then flattens. Adding features beyond the plateau provides negligible benefit while
increasing model complexity — consistent with the authors' finding of **11 optimal features**.

**NaiveBayes plateau is lower (~73–78%):** NB never reaches LinearSVC/MLP accuracy regardless
of feature count, because its independence assumption is violated by the correlated packet-length
cluster identified in §2.7. Adding correlated features doesn't help NB — it may slightly hurt it.

**Reproducibility note:** the brute-force step requires manually reading the plateau from the plot
and setting `OPTIMAL_N`. This was also a manual step in the authors' repo — a reproducibility hazard
documented in Report §4 (Reproducibility Analysis).

**Comparison with the authors' 11 features** (from paper notes):
Bwd Packet Length Std, Average Packet Size, Max Packet Length, Packet Length Variance,
Packet Length Std, Avg Bwd Segment Size, Packet Length Mean, Destination Port,
Init_Win_bytes_forward, Fwd Packet Length Mean, Init_Win_bytes_backward.
Our RF ranking should produce a similar set; any differences reflect the 10% subsample and seed.


### §3.8 — Redundancy Analysis


In [ ]:
# §3.8 — Three complementary redundancy methods:
# (A) r = 1.000 duplicate pairs from §2.7
# (B) RF importance mass distribution across correlated clusters
# (C) VIF for the final selected features

# ── A: Perfect-duplicate pairs ─────────────────────────────────────────────
print('=== A. Perfect-duplicate feature pairs (Spearman r = 1.000, from §2.7) ===\n')
exact_pairs = [
    ('Subflow Fwd Packets',   'Total Fwd Packets',           'CICFlowMeter: single-subflow = total'),
    ('Subflow Fwd Bytes',     'Total Length of Fwd Packets', 'Same artefact'),
    ('Subflow Bwd Packets',   'Total Backward Packets',      'Same artefact'),
    ('Subflow Bwd Bytes',     'Total Length of Bwd Packets', 'Same artefact'),
    ('Avg Fwd Segment Size',  'Fwd Packet Length Mean',      'Two names for identical computation'),
    ('Avg Bwd Segment Size',  'Bwd Packet Length Mean',      'Two names for identical computation'),
    ('Packet Length Std',     'Packet Length Variance',      'Var = Std² (monotone transform)'),
    ('Idle Max',              'Idle Mean',                   'Near-constant idle times'),
    ('Fwd PSH Flags',         'SYN Flag Count',              'CIC capture artefact'),
    ('ECE Flag Count',        'RST Flag Count',              'CIC capture artefact'),
]
present_cols = set(get_feature_cols(df_train_bal))
for a, b, reason in exact_pairs:
    both = (a in present_cols) and (b in present_cols)
    status = '(both present → redundant)' if both else '(one or both dropped in §1)'
    print(f'  {a:<35} ↔ {b:<35} {status}')
    print(f'    Reason: {reason}\n')


In [ ]:
# §3.8 — B: RF importance mass across redundancy clusters.
# The RF naturally distributes importance across correlated features in a cluster;
# the top-N selection keeps the highest-ranked representative of each cluster.
print('=== B. RF importance — redundancy clusters ===\n')
clusters = {
    'Subflow duplicates': [
        'Subflow Fwd Packets', 'Subflow Fwd Bytes',
        'Subflow Bwd Packets', 'Subflow Bwd Bytes',
    ],
    'Packet length / segment size': [
        'Avg Fwd Segment Size', 'Fwd Packet Length Mean',
        'Avg Bwd Segment Size', 'Bwd Packet Length Mean',
        'Packet Length Std',    'Packet Length Variance',
        'Max Packet Length',    'Min Packet Length',
        'Packet Length Mean',   'Average Packet Size',
    ],
    'IAT cluster': [
        'Bwd IAT Total', 'Fwd IAT Total', 'Flow IAT Max',
        'Bwd IAT Max',   'Fwd IAT Max',   'Flow IAT Mean',
    ],
}
for cluster_name, members in clusters.items():
    present = [f for f in members if f in full_importance.index]
    if not present:
        continue
    total_imp = full_importance[present].sum()
    print(f'  {cluster_name}  (combined importance: {total_imp:.4f})')
    for feat in sorted(present, key=lambda x: -full_importance[x]):
        rank_n = list(full_importance.index).index(feat) + 1
        star = '★ SELECTED' if feat in selected_features else '  '
        print(f'    {star}  Rank {rank_n:2d}  imp={full_importance[feat]:.4f}  {feat}')
    print()


In [ ]:
# §3.8 — C: VIF for the final selected features
print('=== C. Variance Inflation Factor — final selected features ===\n')
print('Computing VIF on 10 k subsample...')
X_vif = X_bal_sel.sample(min(10_000, len(X_bal_sel)), random_state=SEED)
vif_df = compute_vif(X_vif)
print(vif_df.to_string(index=False, float_format='{:.2f}'.format))

high_vif = vif_df[vif_df['VIF'] > 5]
print(f'\nFeatures with VIF > 5 (notable multicollinearity): {len(high_vif)}')
if not high_vif.empty:
    for _, row in high_vif.iterrows():
        print(f'  {row["feature"]:<45}  VIF={row["VIF"]:.2f}')


**Redundancy analysis — three methods, consistent conclusion:**

**A — Perfect duplicates (r = 1.000):** The four Subflow features and `Avg Segment Size` duplicates
are removed implicitly — the RF assigns their shared information to one representative, leaving the
other near-zero importance and out of the top-N selection.

**B — RF importance allocation:** Within a correlated cluster, importance mass is split across
all members. The top-N selection automatically retains the most discriminative representative
of each cluster (e.g., `Bwd Packet Length Std` from the packet-length cluster).

**C — VIF:** High VIF on the selected packet-length features is expected — they remain correlated
even after selection because RF importance doesn't require independence. For tree-based models
(DT, RF) VIF is irrelevant. For SVM and ANN, the redundant features add some noise but do not
prevent convergence; StandardScaling already mitigates magnitude differences.

**How to tackle redundancy (for report §3):**
| Method | When to use |
|--------|-------------|
| Correlation filtering (\|r\| > 0.95, drop one per pair) | Before any modelling; fast and interpretable |
| RF importance + top-N (our approach) | Implicitly removes low-importance duplicates while keeping discriminative features |
| VIF-based pruning (drop highest-VIF iteratively until VIF < 5) | When model assumptions require independence (e.g., logistic regression) |
| PCA | Eliminates multicollinearity by construction but loses feature interpretability — avoided here since 11 named features aid cybersecurity interpretation |

The 66 → 11 reduction removes the vast majority of redundancy identified in §2.7.


### §3.9 — Save Preprocessed Artefacts to Drive


In [ ]:
# §3.9 — Persist all §3 outputs so Phase 5 (model training) can reload without re-running.
artifacts = {
    'X_bal_sel.joblib':        X_bal_sel,         # balanced + scaled + selected (model training)
    'y_bal.joblib':            y_bal,
    'X_imbal_sel.joblib':      X_imbal_sel,       # real-prevalence + scaled + selected (Phase 8.2)
    'y_imbal.joblib':          y_imbal,
    'X_test_sel.joblib':       X_test_sel,        # test + scaled + selected
    'y_test.joblib':           y_test,
    'scaler.joblib':           scaler,
    'selected_features.joblib': selected_features,
    'bf_results.joblib':       bf_results,        # brute-force curve (for the report)
    'full_importance.joblib':  full_importance,
}
for filename, obj in artifacts.items():
    path = os.path.join(DATA_DIR, filename)
    dump(obj, path)
    print(f'  Saved  {filename}')

print(f'\nAll §3 artefacts saved to Drive.')
print(f'Selected {OPTIMAL_N} features: {selected_features}')


**§3 complete.** Pipeline summary:

| Step | Input | Output | Key decision |
|------|-------|--------|--------------|
| §3.2 Binary relabel | 15+ classes | 2 classes (BENIGN/ATTACK) | Mirror authors |
| §3.3 Balance 1:1 | 81.9% BENIGN | 48,286 each class | Keep real-prevalence copy too |
| §3.4 Encoding | 66 numeric features | No encoding needed | Destination Port stays numeric |
| §3.5 Derived features | 66 base features | +4 features (70 total) | Bwd/Fwd ratio, pkt range, bytes/pkt, win ratio |
| §3.6 Scaling | 70 features | 70 scaled | Fit on balanced train only — no leakage |
| §3.7 Feature selection | 70 scaled features | Top 11 (plateau) | RF importance + brute-force curve |
| §3.8 Redundancy | 11 features | VIF + cluster map | Expected high VIF; tree models unaffected |

→ **§4 Model Training** uses `X_bal_sel` (shape: `(n_train_bal, 11)`) and `y_bal`.


---
## §4 — Model Training

Train DT, RF, SVM, NB, ANN, DNN with GridSearchCV (k=5). Save best models to Drive.

---
## §5 — Evaluation & Reproduction Check

In-distribution evaluation (reproduce Tables 4–6). Progressive evaluation on CSE-CIC-IDS2018 (reproduce Table 7). Side-by-side comparison with paper's numbers.

---
## §6 — Error Analysis

Misclassified examples (FPs and FNs) on the progressive test set. Patterns in errors. Cybersecurity implications.

---
## §7 — Executive Summary

*(Filled after all analysis is complete.)*

---
## §8 — Summing It Up

*(Filled after all analysis is complete.)*